# Treinando modelo

Abrindo base de dados

In [11]:
import pandas

data_frame = pandas.read_csv('Dataset/processed/data.csv')
data_frame

,Unnamed: 0,file_path,version,class,plant,disease
0,0,Dataset/plantvillage_dataset/segmented/Apple__...,segmented,Apple___Apple_scab,Apple,Apple_scab
1,1,Dataset/plantvillage_dataset/segmented/Apple__...,segmented,Apple___Apple_scab,Apple,Apple_scab
2,2,Dataset/plantvillage_dataset/segmented/Apple__...,segmented,Apple___Apple_scab,Apple,Apple_scab
3,3,Dataset/plantvillage_dataset/segmented/Apple__...,segmented,Apple___Apple_scab,Apple,Apple_scab
4,4,Dataset/plantvillage_dataset/segmented/Apple__...,segmented,Apple___Apple_scab,Apple,Apple_scab
...,...,...,...,...,...,...
162911,162911,Dataset/plantvillage_dataset/color/Tomato___he...,color,Tomato___healthy,Tomato,healthy
162912,162912,Dataset/plantvillage_dataset/color/Tomato___he...,color,Tomato___healthy,Tomato,healthy
162913,162913,Dataset/plantvillage_dataset/color/Tomato___he...,color,Tomato___healthy,Tomato,healthy
162914,162914,Dataset/plantvillage_dataset/color/Tomato___he...,color,Tomato___healthy,Tomato,healthy


Definindo função para o pré-processamento de imagens

In [12]:
import numpy
from PIL import Image
import tensorflow

def preprocesar_imagem(path: str) -> None:
    # Carregando imagem e redimensionando para o tamanho esperado pelo ResNet50
    imagem = numpy.array(Image.open(path).convert('RGB').resize((224, 224)))

    # Pré-Processando a imagem de acordo com o ResNet50
    imagem = tensorflow.keras.applications.resnet50.preprocess_input(imagem)

    return imagem

Separando os dados de X e Y

In [13]:
X = data_frame['file_path']
Y = data_frame['class']

Separando os dados de teste e treino

In [14]:
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(X, Y, test_size=0.2, random_state=1024)

Criando modelo

In [15]:
from keras.src.applications.resnet import ResNet50

base_model = ResNet50(weights='imagenet', include_top=False)

Adicionando camadas personalizadas

In [16]:
from keras.src.layers import Dense
from keras.src.layers.pooling.global_average_pooling2d import GlobalAveragePooling2D

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(1024, activation='relu')(x)
predictions = Dense(len(data_frame['class'].unique()), activation='softmax')(x)

Definindo modelo completo

In [17]:
from keras import Model

model = Model(inputs=base_model.input, outputs=predictions)

Congelando as camadas do modelo base

In [18]:
for layer in base_model.layers:
    layer.treinable = False

Compilando o modelo

In [19]:
from keras.src.optimizers import Adam

model.compile(optimizer=Adam(), loss='categorical_crossentropy', metrics=['accuracy'])

Treinando o modelo ResNet50

In [20]:
for caminho_imagem in x_train:
    imagem = preprocesar_imagem(caminho_imagem)
    model.fit(imagem)

ValueError: Exception encountered when calling Functional.call().

[1mInvalid input shape for input Tensor("data:0", shape=(32, 224, 3), dtype=float32) with name 'keras_tensor_178' and path ''. Expected shape (None, None, None, 3), but input has incompatible shape (32, 224, 3)[0m

Arguments received by Functional.call():
  • inputs=tf.Tensor(shape=(32, 224, 3), dtype=float32)
  • training=True
  • mask=None
  • kwargs=<class 'inspect._empty'>